In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
import math
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
COMPETITION = "eds-232-ocean-chemistry-prediction-for-calcofi"

@dataclass(frozen=True)
class ModelConfig:
    architecture: str = "mlp"
    activation: str = "silu"
    width: int = 256
    dropout: float = 0.15
    loss: str = "mse"
    huber_beta: float = 0.1
    learning_rate: float = 0.002
    weight_decay: float = 0.0001
    grad_clip: float | None = 1.0
    skip_bias: bool = True
    batch_size: int = 32
    epochs: int = 800
    min_epochs: int = 300
    patience: int = 200

PLAN = [
    (
        'SiLU MSE',
        ModelConfig(architecture='mlp_skip', width=128, dropout=0.02),
        10, 100, 42, 0.12,
    ),
    (
        'GELU MSE',
        ModelConfig(architecture='mlp_skip', activation='gelu', width=192, dropout=0.03),
        10, 42, 100, 0.12,
    ),
    (
        'LayerNorm MSE',
        ModelConfig(architecture='mlp_norm', width=128, dropout=0.03),
        10, 2026, 2026, 0.088,
    ),
    (
        'SiLU Smooth L1',
        ModelConfig(
            architecture='mlp_skip',
            width=128,
            dropout=0.03,
            loss='huber',
            huber_beta=0.15,
        ),
        10, 777, 777, 0.172,
    ),
    (
        'Wide SiLU Smooth L1',
        ModelConfig(
            architecture='mlp_skip',
            width=192,
            dropout=0.02,
            loss='huber',
            huber_beta=0.18,
        ),
        10, 584, 1234, 0.5,
    ),
]


def set_up_cpu():
    torch.set_num_threads(min(2, os.cpu_count() or 1))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    set_up_cpu()


In [ ]:
def locate_competition_data():
    kaggle_input = Path("/kaggle/input")
    candidates = [
        kaggle_input / "competitions" / COMPETITION,
        kaggle_input / COMPETITION,
    ]
    if kaggle_input.is_dir():
        candidates.extend(sorted(kaggle_input.rglob(COMPETITION)))
    for folder in candidates:
        if all((folder / name).is_file() for name in ("train.csv", "test.csv", "sample_submission.csv")):
            return folder
    raise FileNotFoundError("Kaggle Notebookの Add Input から、この課題のコンペデータを追加してください。")

data_directory = locate_competition_data()


In [ ]:
FEATURES = (
    "Lat_Dec", "Lon_Dec", "NO2uM", "NO3uM", "NH3uM", "R_TEMP", "R_Depth",
    "R_Sal", "R_DYNHT", "R_Nuts", "R_Oxy_micromol.Kg", "PO4uM", "SiO3uM",
    "Salinity1", "Temperature_degC",
)

def load_data(folder):
    train = pd.read_csv(folder / "train.csv")
    test = pd.read_csv(folder / "test.csv")
    sample = pd.read_csv(folder / "sample_submission.csv")
    if not all(column in train and column in test for column in FEATURES):
        raise ValueError("必要な15列が揃っていません。実装課題①のコンペデータを確認してください。")
    if sample.columns.tolist() != ["id", "DIC"]:
        raise ValueError("提出見本の列が id, DIC と一致しません。")
    if not sample["id"].is_unique or not test["id"].is_unique:
        raise ValueError("提出見本またはテストに重複IDがあります。")
    if set(sample["id"]) != set(test["id"]):
        raise ValueError("提出見本とテストのIDが一致しません。")
    return (
        train.loc[:, list(FEATURES)].to_numpy(dtype=np.float64),
        train["DIC"].to_numpy(dtype=np.float32),
        test.loc[:, list(FEATURES)].to_numpy(dtype=np.float64),
        test["id"],
        sample,
    )

features, labels, test_features, test_ids, sample = load_data(data_directory)
print(f"入力 {len(FEATURES)}列・学習 {len(labels)}件・予測 {len(test_ids)}件・実行環境 {device}")


In [ ]:
def get_activation(name):
    if name == "gelu":
        return nn.GELU()
    return nn.SiLU()

class TabularMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = config.width
        act = lambda: get_activation(config.activation)
        if config.architecture == "mlp_norm":
            self.layers = nn.Sequential(
                nn.Linear(len(FEATURES), width), nn.LayerNorm(width), act(),
                nn.Dropout(config.dropout), nn.Linear(width, width), nn.LayerNorm(width), act(),
                nn.Dropout(config.dropout), nn.Linear(width, width // 2), nn.LayerNorm(width // 2),
                act(), nn.Linear(width // 2, 1),
            )
        else:
            self.layers = nn.Sequential(
                nn.Linear(len(FEATURES), width), act(),
                nn.Dropout(config.dropout), nn.Linear(width, width), act(),
                nn.Dropout(config.dropout), nn.Linear(width, width // 2), act(),
                nn.Linear(width // 2, 1),
            )
        use_skip = config.architecture in {"mlp_skip", "mlp_norm"}
        self.skip = nn.Linear(len(FEATURES), 1, bias=config.skip_bias) if use_skip else None

    def forward(self, inputs):
        prediction = self.layers(inputs)
        if self.skip is not None:
            prediction = prediction + self.skip(inputs)
        return prediction.squeeze(-1)

def build_model(config):
    if config.architecture in {"mlp", "mlp_skip", "mlp_norm"}:
        return TabularMLP(config)
    raise ValueError(f"Unknown architecture: {config.architecture}")


In [ ]:
def fold_indices(n_rows, n_folds, split_seed=42):
    permutation = np.random.RandomState(split_seed).permutation(n_rows)
    all_rows = np.arange(n_rows)
    for group in np.array_split(permutation, n_folds):
        validation = np.sort(group)
        training = np.setdiff1d(all_rows, validation)
        yield training, validation


def prepare_inputs(training, validation, test, device):
    means = np.nanmean(training, axis=0)
    if not np.isfinite(means).all():
        raise ValueError("学習データに全行欠損の入力列があります。")
    filled_train = np.where(np.isnan(training), means, training)
    scale = filled_train.std(axis=0)
    scale[scale == 0] = 1.0

    tensors = []
    for values in (training, validation, test):
        filled = np.where(np.isnan(values), means, values)
        tensors.append(torch.as_tensor((filled - means) / scale, dtype=torch.float32, device=device))
    return tensors


def rmse(target, prediction):
    return float(np.sqrt(np.mean((np.asarray(target, dtype=np.float64) - prediction) ** 2)))


def fit_and_predict(config, train_x, train_y, val_x, val_y, test_x, device, seed):
    torch.manual_seed(seed)
    train, val, test = prepare_inputs(train_x, val_x, test_x, device)
    targets = train_y.astype(np.float64)
    target_mean = float(targets.mean())
    target_scale = max(float(targets.std()), 1e-12)
    labels = torch.as_tensor((targets - target_mean) / target_scale, dtype=torch.float32, device=device)

    loader = DataLoader(TensorDataset(train, labels), batch_size=config.batch_size, shuffle=True)
    model = build_model(config).to(device)
    if config.loss == "huber":
        loss_fn = nn.SmoothL1Loss(beta=config.huber_beta)
    else:
        loss_fn = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.epochs, eta_min=0.00001
    )
    val_target = torch.as_tensor(val_y, dtype=torch.float32, device=device)
    best_rmse = math.inf
    best_epoch = 0
    best_state = None
    stopped_epoch = config.epochs

    for epoch in range(1, config.epochs + 1):
        model.train()
        for inputs, target in loader:
            optimizer.zero_grad()
            loss_fn(model(inputs), target).backward()
            if config.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            predicted_dic = model(val) * target_scale + target_mean
            error = (predicted_dic - val_target).square().mean().sqrt().item()
        if error < best_rmse:
            best_rmse = error
            best_epoch = epoch
            best_state = {name: param.detach().clone() for name, param in model.state_dict().items()}
        if config.patience and epoch >= config.min_epochs and epoch - best_epoch >= config.patience:
            stopped_epoch = epoch
            break

    if best_state is None:
        raise RuntimeError("検証スコアを計算できませんでした。")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(val).cpu().numpy() * target_scale + target_mean
        test_pred = model(test).cpu().numpy() * target_scale + target_mean
    if not np.isfinite(val_pred).all() or not np.isfinite(test_pred).all():
        raise RuntimeError("予測に数値として扱えない値が含まれています。")
    return val_pred, test_pred, best_epoch, stopped_epoch


In [ ]:
combined_validation = np.zeros(len(labels), dtype=np.float64)
combined_test = np.zeros(len(test_ids), dtype=np.float64)
total_weight = sum(weight for _, _, _, _, _, weight in PLAN)

for name, config, folds, seed, split_seed, weight in PLAN:
    out_of_fold = np.full(len(labels), np.nan)
    team_predictions = []
    for fold, (training, validation) in enumerate(fold_indices(len(labels), folds, split_seed=split_seed), start=1):
        val_pred, test_pred, best_epoch, trained_epochs = fit_and_predict(
            config, features[training], labels[training], features[validation],
            labels[validation], test_features, device, seed + fold - 1,
        )
        out_of_fold[validation] = val_pred
        team_predictions.append(test_pred)
    team_rmse = rmse(labels, out_of_fold)
    print(f"[{name}] {folds}-Fold OOF RMSE: {team_rmse:.5f}", flush=True)
    fraction = weight / total_weight
    combined_validation += fraction * out_of_fold
    combined_test += fraction * np.mean(team_predictions, axis=0)

print(f"\n==========================================")
print(f"全体の交差検証RMSE: {rmse(labels, combined_validation):.5f}")
print(f"==========================================")


In [ ]:
def make_submission(sample, test_ids, predictions):
    prediction = np.asarray(predictions)
    if prediction.shape != (len(test_ids),) or not np.isfinite(prediction).all():
        raise ValueError("テスト全件の有限な予測が揃っていません。")
    lookup = pd.Series(prediction, index=test_ids.to_numpy())
    result = sample[["id"]].copy()
    result["DIC"] = result["id"].map(lookup)
    if result["DIC"].isna().any():
        raise ValueError("提出見本のIDに対応する予測がありません。")
    return result

submission = make_submission(sample, test_ids, combined_test)
output_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
output_file = output_dir / "submission.csv"
submission.to_csv(output_file, index=False)
print(f"{output_file} を作成しました（{len(submission)}行）")

submission.head()
